# Chapter 21: Composing Blocks into Bigger Models

        **Part III - Building Models from Blocks** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Package repeated layers as reusable blocks
- Compose nested modules
- Implement a residual connection with matching shapes


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## A reusable block

A block gives a recurring transformation one name and one tested shape contract.


In [ ]:
import torch.nn as nn

class DenseBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.LayerNorm(width))

    def forward(self, x):
        return self.net(x)

block = DenseBlock(16)
print(block(torch.randn(4, 16)).shape)


## Nested composition

ModuleList registers a programmable collection of blocks while leaving loop logic in forward.


In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, width=16, depth=3):
        super().__init__()
        self.blocks = nn.ModuleList([DenseBlock(width) for _ in range(depth)])
        self.head = nn.Linear(width, 3)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.head(x)

deep = DeepMLP()
print(deep(torch.randn(5, 16)).shape)
print("parameter tensors:", len(list(deep.parameters())))


## Residual composition

A skip connection adds a block's input to its output. Addition requires identical shapes.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.transform = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return torch.relu(x + self.transform(x))

residual = ResidualBlock(12)
sample = torch.randn(7, 12)
print(sample.shape, residual(sample).shape)


## Practice

        Work through these without looking back first.

        1. Create a block that changes width safely.
2. Build a network from five registered blocks.
3. Explain what must be true before tensors can be added in a residual path.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 21's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
